# TOMOseq data analysis pipeline

In [2]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib_venn as venn3
#import plotly.graph_objs as go
from bioinfokit import visuz
from sklearn.cluster import KMeans
import seaborn as sns
from scipy.stats import zscore, variation
from sklearn.preprocessing import normalize, scale, MinMaxScaler
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import dendrogram, linkage
from mpl_toolkits.axes_grid1 import make_axes_locatable
import random
import multiprocessing
from statsmodels.stats.multitest import multipletests
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import scanpy as scy
import anndata as ad
from venn import venn
%matplotlib inline
import scipy
from collections import defaultdict
from matplotlib.patches import Patch
from scipy.stats import ttest_ind
#from statannotations.Annotator import Annotator
import scanpy.external as sce
import gseapy
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import LogFormatterSciNotation
import statsmodels.formula.api as smf
from tqdm import tqdm
import matplotlib as mpl
import matplotlib.patches as mpatches

from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib as mpl

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [3]:
print(np.__version__)
print(pd.__version__)

1.26.4
2.0.3


Describe funciton to process de data

In [4]:
def accessData(list_files, path, list_labels):
    """opens each file and stores in dictionary with key a label"""
    #create empty dictionary to store the dataframes
    dic_dataframes = {}
    #iterate to get the file names
    #make a counter to access the correct label
    count = 0
    for file in list_files:
        #open each file
        data_frame = pd.read_csv(path+file, sep='\t', index_col=False,  header=None)
        #add the dataframe as an element of a dictionary with key the filename
        dic_dataframes[list_labels[count]] = data_frame
        count = count + 1

    return dic_dataframes

def accessData2(list_files, path, list_labels):
    """opens the read/transcript/gene counts of each file"""
    #create empty dictionary to store the dataframes
    dic_dataframes = {}
    #iterate to get the file names
    #make a counter to access the correct label
    count = 0
    for file in list_files:
        #open each file
        data_frame = pd.read_csv(path+file, sep='\t', index_col=0,  header=None)
        #add the dataframe as an element of a dictionary with key the filename
        dic_dataframes[list_labels[count]] = data_frame
        count = count + 1
    
    return dic_dataframes

def accessData3(list_files, path, list_labels):
    """opens the read/transcript/gene counts of each file"""
    #create empty dictionary to store the dataframes
    dic_dataframes = {}
    #iterate to get the file names
    #make a counter to access the correct label
    count = 0
    for file in list_files:
        #open each file
        data_frame = pd.read_csv(path+file, sep='\t', index_col=0)
        #add the dataframe as an element of a dictionary with key the filename
        dic_dataframes[list_labels[count]] = data_frame
        count = count + 1
    
    return dic_dataframes


def accessData4(list_files, path, list_labels):
    """opens each file and stores in dictionary with key a label"""
    #create empty dictionary to store the dataframes
    dic_dataframes = {}
    #iterate to get the file names
    #make a counter to access the correct label
    count = 0
    for file in list_files:
        #open each file
        data_frame = pd.read_csv(path+file, sep='\t', index_col=[0,1,2,3,4,5,6,7,8,9],  header=0)
        #add the dataframe as an element of a dictionary with key the filename
        dic_dataframes[list_labels[count]] = data_frame
        count = count + 1
    return dic_dataframes

def accessData5(list_files, path, list_labels):
    """opens each file and stores in dictionary with key a label"""
    #create empty dictionary to store the dataframes
    dic_dataframes = {}
    #iterate to get the file names
    #make a counter to access the correct label
    count = 0
    for file in list_files:
        #open each file
        data_frame = pd.read_csv(path+file, sep='\t', index_col=False,  header=0)
        #add the dataframe as an element of a dictionary with key the filename
        dic_dataframes[list_labels[count]] = data_frame
        count = count + 1

    return dic_dataframes

def saveCSV(dic_data, path, label):
    """Save dataframes inside dictionary to csv files"""
    for key in dic_data:
        dic_data[key].to_csv(path+key+label+'.bed', sep='\t', header=False)

    return print('Files saved')

def saveCSV2(dic_data, path, label):
    """Save dataframes inside dictionary to csv files"""
    for key in dic_data:
        dic_data[key].to_csv(path+key+label+'.bed', sep='\t', header=True)

    return print('Files saved')


def normalize(dic_data, dic_total, dic_scaling_value):
    """Normalize to desired value (mean, median...)
    1. divide by the total counts in each column
    2. multiply by scaling value"""
    #make a counter for the smaple number to select the scaling_value position
    count = 0
    #make a dictionary to store the normalized datasets
    dic_normal = {}
    #Get each dataframe
    for key in dic_data:
        #divide each section by the total of the section
        data_frame_normal_s1 = dic_data[key].divide(dic_total[key], axis='columns')
        #multiply by the scaling value
        data_frame_normal = data_frame_normal_s1.multiply(dic_scaling_value[key])
        count = count + 1
        #save the dataframe to the dictionary
        dic_normal[key] = data_frame_normal

    return dic_normal

def zscores(dic_data):
    """Calculates de z scores for each gene individually"""
    #Create a dictinary to store the dataframes of z scores
    dic_zscores = {}
    for key in dic_data:
        #create a dataframe to store the zscores
        df_zscore = pd.DataFrame(columns = dic_data[key].columns, index = dic_data[key].index)
        #calculate z scores for each gene
        #for that we need to get the info from every row independently and then get the z score
        for index in dic_data[key].index.values:
            #transform the series into list, then to np.array and apply z score funtion from scipy.stats
            #and save it to the Zscore df
            data_gene = np.array(list(dic_data[key].loc[index]))
            df_zscore.loc[index] = zscore(data_gene.astype(float))
        #save the df_zscores to the dictionary
        dic_zscores[key] = df_zscore.astype(float)
    
    return dic_zscores


def scaledZscores(dic_data):
    """Calculates de scaled z scores for each gene individually.
    Normalize the Z score to scaled Z score (-1 to 1)"""
    #Create a dictinary to store the dataframes of z scores
    dic_scaled_zscores = {}
    for key in dic_data:
        #create a dataframe to store the zscores
        df_scaled_zscore = pd.DataFrame(columns = dic_data[key].columns, index = dic_data[key].index)
        #calculate z scores for each gene
        #for that we need to get the info from every row independently and then get the z score
        for index in dic_data[key].index.values:
            ## 1.convert the column value of the dataframe to floats
            float_array = dic_data[key].loc[index].values.astype(float)
            # Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.
            float_array = float_array.reshape(-1, 1)
            # 2. create a min max processing object
            min_max_scaler = MinMaxScaler(feature_range=(-1, 1))
            scaled_array = min_max_scaler.fit_transform(float_array)
            # 3. convert the scaled array to dataframe. First reshape the array to the original shape
            scaled_array = scaled_array.reshape(len(dic_data[key].columns.values),)
            df_scaled_zscore.loc[index] = scaled_array
        #save the df_zscores to the dictionary 
        #Use astype(float) to ensure the values are numbers for correct plotting
        dic_scaled_zscores[key] = df_scaled_zscore.astype(float)
    
    return dic_scaled_zscores


def meanDataset(dic_data):
    """Sum the different datasets and calculate the mean to create a Mean dataset"""
    #get a list of the keys
    keys = []
    for key in dic_data:
        keys.append(key)
    #create a new dataframe
    mean_dataset = pd.DataFrame(index = dic_data[keys[0]].index.values, columns = dic_data[keys[0]].index.values)
    for element in range(len(keys)):
        #add each dataframe individually
        #first combine the first two dataframes
        if element == 0:
            mean_dataset = dic_data[keys[0]].add(dic_data[keys[1]], axis='index')
        #add the remaining dataframes to the existent sum
        if element > 1:
            mean_dataset = mean_dataset.add(dic_data[keys[element]], axis='index')
    #divide by the number of samples to make the mean
    mean_dataset = mean_dataset.div(len(keys))
    
    #re-normalize z-scores
    #create dic_data 
    dic_data = {'mean': mean_dataset}
    mean_dataset_scaled = scaledZscores(dic_data)
    
    return mean_dataset_scaled


## Anna's formulas for hierarchical clustering
def hierarchicalClustering(df, cth = 100, plot = False, method = 'ward', metric = 'euclidean', nolabels = 'True', leaf_colors = []):
    """performs hierarchical clustering using linkage and dendogram functions from scipy.cluster.hierarchy package"""
    if len(leaf_colors) > 0:
        hierarchy.set_link_color_palette(leaf_colors)
    Z = linkage(df, method=method, metric = metric)
    dg = dendrogram(Z, no_labels=nolabels, color_threshold=cth, no_plot = np.invert(plot))
    plt.show()
    return Z, dg

## Anna's formulas for hierarchical clustering
def hierarchicalClustering2(df, cth = 100, plot = False, method = 'average', metric = 'euclidean', nolabels = 'True', leaf_colors = []):
    """performs hierarchical clustering using linkage and dendogram functions from scipy.cluster.hierarchy package"""
    if len(leaf_colors) > 0:
        hierarchy.set_link_color_palette(leaf_colors)
    Z = linkage(df, method=method, metric = metric)
    dg = dendrogram(Z, no_labels=nolabels, color_threshold=cth, no_plot = np.invert(plot))
    plt.show()
    return Z, dg

def getClusterByColor(dg, labels):
    """given a dendogram and labels, it groups labels by colors in the dendogram (i.e. clusters)"""
    kk = []
    ii = 0
    cluster = 0
    color = dg['color_list'][0]
    clusters = {cluster: []}
    for i in range(len(dg['icoord'])):
        v = dg['icoord'][i]
        for j in [0,2]:
            vj = int(round((v[j]-5.)/10))
            if (v[j]-5.)/10 == vj and vj not in kk:
                kk.append(vj)
                if dg['color_list'][i] == color:
                    clusters[cluster].append(labels[dg['leaves'][vj]])
                else:
                    color = dg['color_list'][i]
                    cluster += 1
                    clusters[cluster] = [labels[dg['leaves'][vj]]]
    return clusters


def plotHeatmap(dic_data, dg, clusters, figsize, title, type_data, reorganize = False, new_index_order = []):
    """Use the combined clustering to plot the indivudal heatmaps for each sample"""
    #access each dictionary and count the number of samples
    keys = []
    for key in dic_data:
        keys.append(key)      
    #check the number of samples to provide the width ratios
    ratios = [0.5]
    for element in range(len(keys)):
        if element < len(keys)-1:
            ratios.append(3)
        if element == len(keys)-1:
            ratios.append(4)
    # plot multiple plots with plt.sublots. axs is a list of all the subplots. Plot as many plots as samples plus 
    # one for the cluster label
    fig, axs = plt.subplots(nrows = 1, ncols = len(keys)+1, sharey='row', gridspec_kw={'width_ratios': ratios}, figsize = figsize);
    #make a count to access the correct plot
    count = 0
    for key in dic_data:
        if reorganize == False:
            #use dg['leaves'] indexes to organize the data to plot
            data_plot = dic_data[key].loc[dic_data[key].index[dg['leaves']]]
        if reorganize == True:
            data_plot = dic_data[key].reindex(index = new_index_order)
        # to plot I use ax.imshow
        im = axs[count+1].imshow(data_plot, aspect='auto', interpolation='none')
        # add labels
        axs[count+1].set_title(key)
        #remove ticks on the y axis
        axs[count+1].tick_params(top=False, bottom=True, left=False, right=False)
        #add 1 to the count to go to next sample
        count = count + 1
        
    #create the colour bar and label it, we use the last image (im)
    #cax = divider.append_axes('left', size='5%', pad=0.05)
    cb = fig.colorbar(im, orientation='vertical', shrink = 0.3)
    cb.set_label(type_data)

    # Make a big plot in order to craete shared axis labels
    # add a big axes, hide frame
    fig.add_subplot(111, frameon=False)
    # hide tick and tick label of the big axes
    plt.tick_params(labelcolor='none', top=False, bottom=False, left=False, right=False)
    plt.grid(False)
    plt.xlabel("Sections A-P")

    #Introduce the cluster bar
    #make the variable bottom to 0, which indicates were the cluster starts
    bottom = 0
    for cluster in clusters:
        if int(cluster) < len(clusters)-1: 
            #get the height of the cluster
            height = len(clusters[cluster])
            axs[0].bar(-0.5, height = height, bottom=bottom-0.5, width=0.2, align='center')
            #add cluster labels
            axs[0].text(-0.5, bottom-0.5+height/2, str(int(cluster)+1), ha='center', va='center')
            #add the length of the current cluster to the bottom variable for the next cluster start
            bottom = bottom + len(clusters[cluster])

        #I add -1 to the last bottom, otherwise the plot gets out of the grid
        if int(cluster) == len(clusters)-1:
            height = len(clusters[cluster])
            axs[0].bar(-0.5, height = height, bottom=bottom-0.5, width=0.2, align='center')
            #add cluster label
            axs[0].text(-0.5, bottom-0.5+height/2, str(int(cluster)+1), ha='center', va='center')

    #axs[0].set_ylim(0,bottom)  

    #remove ticks and plot frame from the cluster bar
    axs[0].tick_params(labelcolor='none', top=False, bottom=False, left=False, right=False)
    axs[0].axis('off')
    #add the figure title
    fig.suptitle(title, fontsize=13)
    plt.show()
    
    
def ClusterGeneNames(data, clusters, dg, file_name):
    """Creates a txt file that contains the gene ids of each cluster to use for GO search"""
    #create the file
    #We declared the variable f to open a file . Open takes 2 arguments, the file that we want 
    #to open and a string that represents the kinds of permission or operation we want to do on the file
    #Here, we used "w" letter in our argument, which indicates write and will create a file if it does 
    #not exist in library
    #Plus sign indicates both read and write.
    f = open(file_name,"w+")
    #change the index order of the data in the heatmap
    data = data.index[dg['leaves']]
    #access the gene info of each cluster
    start = 0
    end = 0
    for key in clusters:
        #writte the cluster name
        #The output we want to iterate in the file is "Cluster ", which we declare with write function and 
        #then percent d (displays integer)
        #So basically we are putting in the line number that we are writing (key + 1), then putting it in a carriage 
        #return and a new line character
        f.write('\nCluster %d\r\n' % (int(key)+1))
        #go through each index to substract the gene id and write it on the file
        end = end + len(clusters[key])
        for element in data[start:end]:
            #split the index and save the gene id and new line
            info = element.split('_')
            f.write(str(info[1]) + '\n')
        start = start + len(clusters[key])
    #close the file when done
    f.close()
    
    return print(file_name)


def ClusterEnsambleID(data, clusters, dg, file_name):
    """Creates a txt file that contains the gene ids of each cluster to use for GO search"""
    #create the file
    #We declared the variable f to open a file . Open takes 2 arguments, the file that we want 
    #to open and a string that represents the kinds of permission or operation we want to do on the file
    #Here, we used "w" letter in our argument, which indicates write and will create a file if it does 
    #not exist in library
    #Plus sign indicates both read and write.
    f = open(file_name,"w+")
    #change the index order of the data in the heatmap
    data = data.index[dg['leaves']]
    #access the gene info of each cluster
    start = 0
    end = 0
    for key in clusters:
        #writte the cluster name
        #The output we want to iterate in the file is "Cluster ", which we declare with write function and 
        #then percent d (displays integer)
        #So basically we are putting in the line number that we are writing (key + 1), then putting it in a carriage 
        #return and a new line character
        f.write('\nCluster %d\r\n' % (int(key)+1))
        #go through each index to substract the gene id and write it on the file
        end = end + len(clusters[key])
        for element in data[start:end]:
            #split the index and save the gene id and new line
            info = element.split('_')
            f.write(str(info[0]) + '\n')
        start = start + len(clusters[key])
    #close the file when done
    f.close()
    
    return print(file_name)


def ClusterIndexs(data, clusters, dg, file_name):
    """Creates a txt file that contains the gene ids of each cluster to use for GO search"""
    #create the file
    #We declared the variable f to open a file . Open takes 2 arguments, the file that we want 
    #to open and a string that represents the kinds of permission or operation we want to do on the file
    #Here, we used "w" letter in our argument, which indicates write and will create a file if it does 
    #not exist in library
    #Plus sign indicates both read and write.
    f = open(file_name,"w+")
    #create new dictiornary
    idx_cluster={}
    #change the index order of the data in the heatmap
    data = data.index[dg['leaves']]
    #access the gene info of each cluster
    start = 0
    end = 0
    for key in clusters:
        #writte the cluster name
        #The output we want to iterate in the file is "Cluster ", which we declare with write function and 
        #then percent d (displays integer)
        #So basically we are putting in the line number that we are writing (key + 1), then putting it in a carriage 
        #return and a new line character
        f.write('\nCluster %d\r\n' % (int(key)+1))
        #go through each index to substract the gene id and write it on the file
        end = end + len(clusters[key])
        list_idx=[]
        for index in data[start:end]:
            f.write(str(index) + '\n')
            list_idx.append(str(index))
        start = start + len(clusters[key])
        idx_cluster[str(key)]=list_idx
    #close the file when done
    f.close()
    
    return idx_cluster, print(file_name)


def ClusterIndex(data, clusters, dg, file_name):
    """Creates a txt file that contains the gene ids of each cluster to use for GO search"""
    #create the file
    #We declared the variable f to open a file . Open takes 2 arguments, the file that we want 
    #to open and a string that represents the kinds of permission or operation we want to do on the file
    #Here, we used "w" letter in our argument, which indicates write and will create a file if it does 
    #not exist in library
    #Plus sign indicates both read and write.
    f = open(file_name,"w+")
    dic_data={}
    #change the index order of the data in the heatmap
    data = data.index[dg['leaves']]
    #access the gene info of each cluster
    start = 0
    end = 0
    for key in clusters:
        #writte the cluster name
        #The output we want to iterate in the file is "Cluster ", which we declare with write function and 
        #then percent d (displays integer)
        #So basically we are putting in the line number that we are writing (key + 1), then putting it in a carriage 
        #return and a new line character
        f.write('\nCluster %d\r\n' % (int(key)+1))
        #go through each index to substract the gene id and write it on the file
        end = end + len(clusters[key])
        list_index=[]
        for element in data[start:end]:
            f.write(element + '\n')
            list_index.append(element)
        start = start + len(clusters[key])
        dic_data[key]=list_index
    #close the file when done
    f.close()
    
    return print(file_name), dic_data


def plotMeanCluster(dic_data, list_clusters, xlabel = 'Sections A-P', ylabel = 'Scaled z-score'):
    """Plot the specified cluster data"""
    #get cluster data and plot
    for key in dic_data:
        data_frame = dic_data[key]
        #find the column values for the cluster of interest
        for cluster in list_clusters:
            cluster_data = data_frame.loc[str(cluster)]
            #plot the data
            cluster_data.plot(label = cluster+1);
        plt.xlabel(xlabel,fontsize=15);
        plt.ylabel(ylabel,fontsize=15);
        plt.legend(fontsize=15, frameon=False);
        plt.title(key,fontsize=15)
        plt.xticks(fontsize=15)
        plt.yticks(fontsize=15)
        plt.show();
    
    return plt

## 1- Integration of regionalization from controls and mdx

In [8]:
#access de data
#Create a variable list with the file names and one with the labels to use as dictionary keys
path=''
file_names = ['M1M2M3_1000bp0ovl_cpg_coverage_table_filtered_normal_TSS3kb.bed']
labels = ['shared_filtered_merged']


#use accessData() function to obtain a dictonary with each dataset with labels as key
data_filtered = accessData2(file_names, path, labels)
data_filtered['shared_filtered_merged']

/var/folders/mg/6wjwkry15y79vlj0f727ljh40000gq/T/ipykernel_3020/1401350171.py:26: DtypeWarning: Columns (0,38) have mixed types. Specify dtype option on import or set low_memory=False.
  data_frame = pd.read_csv(path+file, sep='\t', index_col=0,  header=None)


,1,2,3,4,5,6,7,8,9,10,...,33,34,35,36,37,38,39,40,41,42
0,,,,,,,,,,,,,,,,,,,,,
1,3069826,3070825,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,0.000000,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3070826,3071825,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.899105,0.000000,0.000000,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3071826,3072825,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.730564,0.0,0.000000,0.000000,0.000000,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3072826,3073825,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,7.192841,1.817664,0.962274,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3098826,3099825,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,0.000000,1,3099016,3102116,+,ENSMUSG00000064842_Gm26206_snRNA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Y,10613510,10614509,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,0.000000,Y,10613361,10616461,+,ENSMUSG00000118422_Gm21900_UnprocessedPseudogene
Y,10615510,10616509,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,0.000000,Y,10613361,10616461,+,ENSMUSG00000118422_Gm21900_UnprocessedPseudogene
Y,10623510,10624509,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,0.000000,Y,10623743,10626843,+,ENSMUSG00000100369_Gm29527_lincRNA


In [9]:
#drop Y and X chromosomes
data_filtered['shared_filtered_merged']=data_filtered['shared_filtered_merged'].drop(index=["X", "Y"])
data_filtered['shared_filtered_merged']

,1,2,3,4,5,6,7,8,9,10,...,33,34,35,36,37,38,39,40,41,42
0,,,,,,,,,,,,,,,,,,,,,
1,3069826,3070825,5,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3070826,3071825,1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.899105,0.000000,0.000000,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3071826,3072825,3,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,1.730564,0.000000,0.000000,0.000000,0.000000,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3072826,3073825,4,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,7.192841,1.817664,0.962274,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3098826,3099825,5,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1,3099016,3102116,+,ENSMUSG00000064842_Gm26206_snRNA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,124471380,124472379,1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,9,124468703,124471803,+,ENSMUSG00000111523_Gm47183_TEC
9,124476380,124477379,11,0.0,0.0,94.895227,172.163012,181.081654,150.48167,90.916633,...,141.906221,104.726612,159.141614,109.059846,127.982472,9,124476798,124479898,-,ENSMUSG00000093803_Ppp2r3d_ProteinCoding
9,124477380,124478379,1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,9,124476798,124479898,-,ENSMUSG00000093803_Ppp2r3d_ProteinCoding


In [10]:
#rename the header
data_filtered['shared_filtered_merged'].columns = ['start', 'end', 'Lpnp1', 'M1_s10', 'M1_s11', 'M1_s12', 'M1_s13',
       'M1_s24', 'M1_s25', 'M1_s27', 'M1_s35', 'M1_s36',
       'M1_s38', 'M2_s8', 'M2_s9', 'M2_s10', 'M2_s11', 'M2_s22', 'M2_s23',
       'M2_s24', 'M2_s25', 'M2_s36', 'M2_s37', 'M2_s38', 'M2_s39', 'M3_s1',
       'M3_s2', 'M3_s3', 'M3_s4', 'M3_s17', 'M3_s18', 'M3_s19', 'M3_s20',
       'M3_s33', 'M3_s34', 'M3_s35', 'M3_s36', 'chr_g', 'start_g', 'end_g', 'strand_g', 'gene']
data_filtered['shared_filtered_merged']

,start,end,Lpnp1,M1_s10,M1_s11,M1_s12,M1_s13,M1_s24,M1_s25,M1_s27,...,M3_s20,M3_s33,M3_s34,M3_s35,M3_s36,chr_g,start_g,end_g,strand_g,gene
0,,,,,,,,,,,,,,,,,,,,,
1,3069826,3070825,5,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3070826,3071825,1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.899105,0.000000,0.000000,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3071826,3072825,3,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,1.730564,0.000000,0.000000,0.000000,0.000000,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3072826,3073825,4,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,7.192841,1.817664,0.962274,1,3070253,3073353,+,ENSMUSG00000102693_4933401J01Rik_TEC
1,3098826,3099825,5,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1,3099016,3102116,+,ENSMUSG00000064842_Gm26206_snRNA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,124471380,124472379,1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,9,124468703,124471803,+,ENSMUSG00000111523_Gm47183_TEC
9,124476380,124477379,11,0.0,0.0,94.895227,172.163012,181.081654,150.48167,90.916633,...,141.906221,104.726612,159.141614,109.059846,127.982472,9,124476798,124479898,-,ENSMUSG00000093803_Ppp2r3d_ProteinCoding
9,124477380,124478379,1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,9,124476798,124479898,-,ENSMUSG00000093803_Ppp2r3d_ProteinCoding


In [11]:
data_filtered['shared_filtered_merged']=data_filtered['shared_filtered_merged'].drop(['start_g', 'end_g', 'Lpnp1', 'strand_g'], axis=1)
# merge columns 'col1', 'col2', 'col3' into a new column 'merged'
merged_data = data_filtered['shared_filtered_merged'][['chr_g', 'start', 'end', 'gene']].astype(str).agg('_'.join, axis=1)
data_filtered['shared_filtered_merged']['CpG']=merged_data
data_filtered['shared_filtered_merged']=data_filtered['shared_filtered_merged'].drop(['start', 'end', 'chr_g', 'gene'], axis=1)
data_filtered['shared_filtered_merged']


,M1_s10,M1_s11,M1_s12,M1_s13,M1_s24,M1_s25,M1_s27,M1_s35,M1_s36,M1_s38,...,M3_s4,M3_s17,M3_s18,M3_s19,M3_s20,M3_s33,M3_s34,M3_s35,M3_s36,CpG
0,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1_3069826_3070825_ENSMUSG00000102693_4933401J0...
1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.668406,0.000000,0.000000,0.000000,0.000000,0.899105,0.000000,0.000000,1_3070826_3071825_ENSMUSG00000102693_4933401J0...
1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.730564,0.000000,0.000000,0.000000,0.000000,1_3071826_3072825_ENSMUSG00000102693_4933401J0...
1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.826913,0.000000,0.000000,7.192841,1.817664,0.962274,1_3072826_3073825_ENSMUSG00000102693_4933401J0...
1,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1_3098826_3099825_ENSMUSG00000064842_Gm26206_s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9_124471380_124472379_ENSMUSG00000111523_Gm471...
9,0.0,0.0,94.895227,172.163012,181.081654,150.48167,90.916633,78.314717,20.666924,121.898502,...,248.005479,119.291018,109.190558,126.517638,141.906221,104.726612,159.141614,109.059846,127.982472,9_124476380_124477379_ENSMUSG00000093803_Ppp2r...
9,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9_124477380_124478379_ENSMUSG00000093803_Ppp2r...


In [12]:
# keep rows where at least one column is non-zero
df = data_filtered['shared_filtered_merged'].drop(['CpG'], axis=1)
# create mask: count how many columns are >= 4 per row, keep rows with at least 4 sections with coverage
mask = (df >= 4).sum(axis=1) >= 4
df_filtered = data_filtered['shared_filtered_merged'].loc[mask]
df_filtered


,M1_s10,M1_s11,M1_s12,M1_s13,M1_s24,M1_s25,M1_s27,M1_s35,M1_s36,M1_s38,...,M3_s4,M3_s17,M3_s18,M3_s19,M3_s20,M3_s33,M3_s34,M3_s35,M3_s36,CpG
0,,,,,,,,,,,,,,,,,,,,,
1,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,21.754657,0.000000,...,24.611231,16.684058,1.761138,3.307651,8.652818,1.821332,16.183893,32.717954,1.924548,1_3101826_3102825_ENSMUSG00000064842_Gm26206_s...
1,0.000000,0.00000,0.000000,3.957770,0.000000,0.000000,0.000000,4.350818,0.000000,0.000000,...,0.000000,0.834203,5.283414,0.000000,3.461127,7.285330,0.899105,0.000000,0.000000,1_3248826_3249825_ENSMUSG00000102851_Gm18956_P...
1,0.000000,0.00000,34.834957,0.000000,0.000000,0.000000,46.349656,1.087704,0.000000,0.000000,...,12.305615,5.005218,1.761138,4.961476,11.248664,2.731999,6.293736,13.632481,12.509565,1_3252826_3253825_ENSMUSG00000102851_Gm18956_P...
1,0.000000,0.00000,2.402411,0.000000,2.874312,11.714142,0.000000,0.000000,0.000000,0.000000,...,0.000000,6.673623,0.880569,0.000000,0.000000,0.910666,0.000000,0.908832,1.924548,1_3369826_3370825_ENSMUSG00000103377_Gm37180_TEC
1,0.000000,0.00000,4.804822,11.873311,0.000000,0.000000,0.000000,27.192610,2.175466,4.799154,...,3.786343,3.336812,1.761138,6.615301,6.056973,9.106662,2.697315,18.176641,4.811371,1_3378826_3379825_ENSMUSG00000104017_Gm37363_TEC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,480.163773,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,80.490126,0.000000,0.000000,...,0.000000,1.668406,0.880569,1.653825,0.000000,0.000000,1.798210,0.000000,7.698194,9_123804380_123805379_ENSMUSG00000048521_Cxcr6...
9,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,7.442214,0.865282,5.463997,0.000000,0.908832,0.000000,9_124282380_124283379_ENSMUSG00000110834_Gm394...
9,0.000000,111.74239,0.000000,0.000000,0.000000,0.000000,8.913395,0.000000,21.754657,0.000000,...,1.893172,5.005218,9.686259,12.403690,12.979227,10.927994,5.394631,12.723649,6.735920,9_124347380_124348379_ENSMUSG00000111733_Gm471...


In [13]:
#transpose dataframe
sample_metadata=pd.DataFrame(index=df_filtered.columns[:-1])
sample_metadata['Condition']=['female']*22 + ['male']*12
sample_metadata['annotation']=['central']*7+['proximal-distal']*3+ ['central']*8+['proximal-distal']*8+ ['central']*4+ ['proximal-distal']*4
sample_metadata


,Condition,annotation
M1_s10,female,central
M1_s11,female,central
M1_s12,female,central
M1_s13,female,central
M1_s24,female,central
M1_s25,female,central
M1_s27,female,central
M1_s35,female,proximal-distal
M1_s36,female,proximal-distal
M1_s38,female,proximal-distal


In [14]:
# Assuming your expression table is called df_counts
# Keep only the columns with actual sample counts
sample_cols = sample_metadata.index.tolist()  # match column names to metadata

df_counts=df_filtered

# melt the table
df_long = df_counts.reset_index().melt(
    id_vars=['CpG'],  # or the column in your table that identifies the gene
    value_vars=sample_cols,
    var_name='Sample',
    value_name='Expression'
)


In [15]:
# sample_metadata should have 'Condition' and 'annotation'
df_long = df_long.merge(
    sample_metadata[['Condition', 'annotation']],
    left_on='Sample',
    right_index=True
)
df_long

,CpG,Sample,Expression,Condition,annotation
0,1_3101826_3102825_ENSMUSG00000064842_Gm26206_s...,M1_s10,0.000000,female,central
1,1_3248826_3249825_ENSMUSG00000102851_Gm18956_P...,M1_s10,0.000000,female,central
2,1_3252826_3253825_ENSMUSG00000102851_Gm18956_P...,M1_s10,0.000000,female,central
3,1_3369826_3370825_ENSMUSG00000103377_Gm37180_TEC,M1_s10,0.000000,female,central
4,1_3378826_3379825_ENSMUSG00000104017_Gm37363_TEC,M1_s10,0.000000,female,central
...,...,...,...,...,...
3994077,9_123804380_123805379_ENSMUSG00000048521_Cxcr6...,M3_s36,7.698194,male,proximal-distal
3994078,9_124282380_124283379_ENSMUSG00000110834_Gm394...,M3_s36,0.000000,male,proximal-distal
3994079,9_124347380_124348379_ENSMUSG00000111733_Gm471...,M3_s36,6.735920,male,proximal-distal
3994080,9_124347380_124348379_ENSMUSG00000079741_Nlrp4...,M3_s36,6.735920,male,proximal-distal


## DEG - Model the interaction by condition (male vs female)

### Principles of a Linear Model with Interaction
A linear model with interaction evaluates not just the individual (main) effects of variables on a response (e.g., gene expression), but also whether the effect of one variable depends on the level of another.

In statsmodels or R, interaction terms are denoted like:

- Expression ~ Condition * annotation

This is equivalent to:
- Expression ~ Condition + annotation + Sample:annotation

Where:
- Condition and annotation are categorical variables (e.g., mouse genotype and spatial region),
- Condition:annotation is the interaction term — testing whether the difference between, say, Central and Proximal-distal regions depends on the Sample type.


| Component              | What It Tells You                                                               |
| ---------------------- | ------------------------------------------------------------------------------- |
| **Condition**          | Overall difference between conditions regardless of region                      |
| **annotation**         | Overall difference between regions regardless of genotype                       |
| **Sample\:annotation** | Whether the **region effect** changes depending on the genotype (or vice versa) |


### Are there genes whose Disease vs Control effect is different in Central vs Proximal-distal?

This requires testing for a Group × Region interaction — i.e., you want genes where the disease effect changes depending on annotation.

Use a linear model for each gene, like:

- Expression ~ Condition * annotation

The interaction term (Condition:annotation) tells you whether the effect of Disease differs across regions.

- With Condition = Ctr (baseline) vs mdx, and annotation = Central (baseline) vs Proximal-distal. 
    - if the pval is significant, then there is an effect of annotation (spatial organization) in the expression levels of the gene between Pax7 and DMD.

#####  Note! the reference is only based on alfabetical order, in our case Ctr comes before mdx sample (if you need to change the reference consider the labels of your samples (or look into python documentation to change settings). 

So the interaction term Condition[T.mdx]:annotation[T.Proximal-distal] is testing:

👉 "Is the effect of mdx vs control different in Proximal-distal compared to Central?"

### Interpretation - What does a significant interaction term mean?
The interaction tests whether the difference in expression between mdx and Ctr varies depending on the spatial region.
How can we interpret this?
- The effect of Condition (mdx vs Ctr) is different in Proximal-distal region compared to Central (baseline region).
- The difference between mdx and control changes depending on whether you are in Central vs Proximal-distal.

So when Gene 1 has a significant interaction:
- It means --> The difference between mdx and control changes depending on whether you are in Central vs Proximal-distal.

What it does NOT tell you directly:
- It does not tell you whether mdx Central is higher than control Central — that’s the main effect of Condition.
- It does not tell you whether mdx Proximal-distal is higher than control Proximal-distal — you’d need to calculate that contrast.
- It does not tell you about the overall average difference between mdx and control — only about the change in difference across regions.

Because the interaction only says:

- The difference (mdx − control) is not the same in Central and Proximal-distal.

We could have two alternatives:
- Genes significant in the interaction and also in global may have regionally stronger effects. DEG between mdx and ctr, but stronger DEG when taking into account grupped sections by annotation.
- Genes significant in the interaction but not in the global disease effect are likely spatially modulated. Only DEG when looking by annotation.


In [1]:
#done on OLS_TSS_1kbBins.py

In [16]:
#reopen results file
results_df = pd.read_excel("OLS_TSS.xlsx", index_col=0)

#correct pvals
# Adjust p-values
results_df['fdr pval model'] = multipletests(results_df['pval model'], method='fdr_bh')[1]
results_df['fdr pval_condition'] = multipletests(results_df['pval_condition'], method='fdr_bh')[1]
results_df['fdr pval_annotation'] = multipletests(results_df['pval_annotation'], method='fdr_bh')[1]
results_df['fdr pval_interaction'] = multipletests(results_df['pval_interaction'], method='fdr_bh')[1]

#filter significant tests
results_df_filtered = results_df[results_df["fdr pval model"] <= 0.05]
results_df_filtered


,CpG,pval model,coef_condition,pval_condition,coef_annotation,pval_annotation,coef_interaction,pval_interaction,fdr pval model,fdr pval_condition,fdr pval_annotation,fdr pval_interaction
8,1_3670826_3671825_ENSMUSG00000051951_Xkr4_Prot...,2.960644e-04,6.970095,0.047401,-1.249268e-01,0.963965,5.299991,0.256336,5.097800e-03,0.251195,0.999733,0.999989
27,1_4607826_4608825_ENSMUSG00000102735_Gm7369_Pr...,9.514782e-04,2.910881,0.053208,-1.351951e+00,0.259571,2.454534,0.221282,1.226982e-02,0.269385,0.870427,0.999989
28,1_4609826_4610825_ENSMUSG00000102735_Gm7369_Pr...,6.124617e-05,39.114555,0.000379,-4.793652e+00,0.551022,-4.145355,0.757040,1.561609e-03,0.009976,0.897691,0.999989
38,1_4768826_4769825_ENSMUSG00000103922_Gm6123_Pr...,4.109292e-03,3.739989,0.011925,1.371187e-01,0.904765,-0.281382,0.883141,3.555370e-02,0.103749,0.987250,0.999989
46,1_4807826_4808825_ENSMUSG00000025903_Lypla1_Pr...,6.290766e-03,11.275920,0.005072,7.957031e-01,0.794841,-3.813909,0.457391,4.857879e-02,0.059165,0.962093,0.999989
...,...,...,...,...,...,...,...,...,...,...,...,...
120852,Y_2073510_2074509_ENSMUSG00000100326_Gm28974_P...,6.117628e-05,4.237171,0.000457,-1.821846e-16,1.000000,-0.544534,0.712369,1.560156e-03,0.011386,1.000000,0.999989
120853,Y_2261510_2262509_ENSMUSG00000099614_Uba1y.ps2...,8.162784e-05,9.863225,0.008351,-5.925713e-16,1.000000,3.498911,0.466659,1.920813e-03,0.082410,1.000000,0.999989
120854,Y_2262510_2263509_ENSMUSG00000099614_Uba1y.ps2...,1.519692e-04,7.526647,0.000117,-1.807291e-16,1.000000,-3.068828,0.194181,3.038303e-03,0.004365,1.000000,0.999989
120855,Y_5320510_5321509_ENSMUSG00000101769_Gm29356_P...,6.901073e-07,8.088471,0.000117,-3.806824e-16,1.000000,0.722594,0.773018,4.636148e-05,0.004357,1.000000,0.999989


In [14]:
results_df_filtered = results_df_filtered[~results_df_filtered["CpG"].str.startswith("Y_")]
results_df_filtered = results_df_filtered[~results_df_filtered["CpG"].str.startswith("X_")]
results_df_filtered

,CpG,pval model,coef_condition,pval_condition,coef_annotation,pval_annotation,coef_interaction,pval_interaction,fdr pval model,fdr pval_condition,fdr pval_annotation,fdr pval_interaction
8,1_3670826_3671825_ENSMUSG00000051951_Xkr4_Prot...,2.960644e-04,6.970095,4.740112e-02,-1.249268e-01,0.963965,5.299991,0.256336,5.097800e-03,0.251195,0.999733,0.999989
27,1_4607826_4608825_ENSMUSG00000102735_Gm7369_Pr...,9.514782e-04,2.910881,5.320834e-02,-1.351951e+00,0.259571,2.454534,0.221282,1.226982e-02,0.269385,0.870427,0.999989
28,1_4609826_4610825_ENSMUSG00000102735_Gm7369_Pr...,6.124617e-05,39.114555,3.792898e-04,-4.793652e+00,0.551022,-4.145355,0.757040,1.561609e-03,0.009976,0.897691,0.999989
38,1_4768826_4769825_ENSMUSG00000103922_Gm6123_Pr...,4.109292e-03,3.739989,1.192470e-02,1.371187e-01,0.904765,-0.281382,0.883141,3.555370e-02,0.103749,0.987250,0.999989
46,1_4807826_4808825_ENSMUSG00000025903_Lypla1_Pr...,6.290766e-03,11.275920,5.071512e-03,7.957031e-01,0.794841,-3.813909,0.457391,4.857879e-02,0.059165,0.962093,0.999989
...,...,...,...,...,...,...,...,...,...,...,...,...
117417,9_123022380_123023379_ENSMUSG00000110945_Gm985...,2.054378e-03,7.854458,6.669088e-03,7.813262e-02,0.971800,-0.701715,0.849289,2.148732e-02,0.070964,1.000000,0.999989
117437,9_123215380_123216379_ENSMUSG00000035498_Cdcp1...,1.010689e-05,86.795414,9.391460e-05,3.302428e+01,0.043570,-37.356446,0.163884,3.859363e-04,0.003744,0.833933,0.999989
117446,9_123317380_123318379_ENSMUSG00000058492_Scp2....,1.580426e-06,9.208290,6.183233e-05,-2.552267e-01,0.875068,-0.361620,0.893893,8.967397e-05,0.002760,0.981777,0.999989
117451,9_123526380_123527379_ENSMUSG00000025240_Sacm1...,1.320938e-04,6.852201,3.317427e-02,-4.918329e-16,1.000000,4.891186,0.249988,2.739664e-03,0.200257,1.000000,0.999989


===========================================================
INTERPRETATION OF COEFFICIENTS
Model: Expression ~ Condition * annotation
===========================================================

Reference (baseline) categories:
  - Condition baseline = female
  - annotation baseline = central

Therefore the model is:

  Expression =
        β0
      + β1 * (Condition == male)
      + β2 * (annotation == proximal-distal)
      + β3 * (Condition == male AND annotation == proximal-distal)



-----------------------------------------------------------
### 1. Intercept (β0)
-----------------------------------------------------------
Interpretation:
  Mean Expression for:
    - Condition = female
    - annotation = central

In words:
  β0 = average Expression in "female + central".



-----------------------------------------------------------
### 2. coef_condition (β1) = Condition[T.male]
-----------------------------------------------------------
Comparison:
  (male + central)   VS   (female + central)

Interpretation:
  How much Expression in "male central" differs from
  "female central".

  β1 > 0  → males have higher expression (in central)
  β1 < 0  → males have lower expression (in central)



-----------------------------------------------------------
### 3. coef_annotation (β2) = annotation[T.proximal-distal]
-----------------------------------------------------------
Comparison:
  (female + proximal-distal)   VS   (female + central)

Interpretation:
  How much Expression changes when moving from
  "central" → "proximal-distal" in females.

  β2 > 0  → proximal-distal > central (in females)
  β2 < 0  → proximal-distal < central (in females)



-----------------------------------------------------------
### 4. coef_interaction (β3)
   = Condition[T.male]:annotation[T.proximal-distal]
-----------------------------------------------------------
Interpretation:
  Whether the male–female difference depends on the 
  annotation group.

More precisely:
  β3 = how much the male–female difference changes when
        switching from "central" to "proximal-distal".

  β3 > 0  → the male–female difference is larger 
            in proximal-distal than in central
  β3 < 0  → the male–female difference is smaller 
            (or reversed) in proximal-distal



-----------------------------------------------------------
Summary Table
-----------------------------------------------------------

| Coefficient | Meaning | Comparison |
|------------|----------|------------|
| **β0 (Intercept)** | Baseline mean expression | female + central |
| **β1 (Condition[T.male])** | Effect of male (when annotation = central) | (male central) − (female central) |
| **β2 (annotation[T.proximal-distal])** | Effect of proximal–distal (when Condition = female) | (female proximal-distal) − (female central) |
| **β3 (interaction)** | Change in male–female difference when switching central → proximal-distal | Additional effect in (male proximal-distal) beyond β1 + β2 |




In [15]:
#filter significant tests
results_df_filtered_DEG_male = results_df_filtered[results_df_filtered["fdr pval_condition"] <= 0.05]
results_df_filtered_DEG_male

,CpG,pval model,coef_condition,pval_condition,coef_annotation,pval_annotation,coef_interaction,pval_interaction,fdr pval model,fdr pval_condition,fdr pval_annotation,fdr pval_interaction
28,1_4609826_4610825_ENSMUSG00000102735_Gm7369_Pr...,6.124617e-05,39.114555,3.792898e-04,-4.793652e+00,0.551022,-4.145355,0.757040,1.561609e-03,0.009976,0.897691,0.999989
49,1_4857826_4858825_ENSMUSG00000033813_Tcea1_Pro...,4.987745e-03,8.212569,9.303640e-04,-8.834212e-01,0.630707,-6.551575,0.039178,4.101265e-02,0.018834,0.919782,0.999989
65,1_5276826_5277825_ENSMUSG00000104352_Gm7182_Pr...,1.485950e-04,3.644471,8.552730e-04,-1.565795e-16,1.000000,-0.470432,0.727363,2.994123e-03,0.017785,1.000000,0.999989
116,1_7663826_7664825_ENSMUSG00000103903_Rps2.ps2_...,1.340251e-09,36.171022,2.019928e-08,-2.143125e+00,0.586292,-8.800108,0.186326,2.461682e-07,0.000005,0.906632,0.999989
145,1_9600826_9601825_ENSMUSG00000067879_Vxn_Prote...,1.306821e-03,85.172298,2.029107e-03,2.998090e+01,0.153992,-41.416610,0.235867,1.548113e-02,0.031900,0.868377,0.999989
...,...,...,...,...,...,...,...,...,...,...,...,...
117349,9_122399380_122400379_ENSMUSG00000093598_A7300...,1.433546e-04,19.067037,1.911405e-04,2.481516e+00,0.501691,-7.622284,0.220745,2.913849e-03,0.006180,0.885501,0.999989
117386,9_122845380_122846379_ENSMUSG00000107504_Gm355...,1.382645e-06,8.130908,6.053225e-07,-7.609391e-01,0.474199,-4.312506,0.019908,8.068676e-05,0.000080,0.880453,0.999989
117437,9_123215380_123216379_ENSMUSG00000035498_Cdcp1...,1.010689e-05,86.795414,9.391460e-05,3.302428e+01,0.043570,-37.356446,0.163884,3.859363e-04,0.003744,0.833933,0.999989
117446,9_123317380_123318379_ENSMUSG00000058492_Scp2....,1.580426e-06,9.208290,6.183233e-05,-2.552267e-01,0.875068,-0.361620,0.893893,8.967397e-05,0.002760,0.981777,0.999989


In [16]:
#filter significant tests
results_df_filtered_DEG_male = results_df_filtered[results_df_filtered["fdr pval_condition"] <= 0.05]
results_df_filtered_DEG_male

,CpG,pval model,coef_condition,pval_condition,coef_annotation,pval_annotation,coef_interaction,pval_interaction,fdr pval model,fdr pval_condition,fdr pval_annotation,fdr pval_interaction
28,1_4609826_4610825_ENSMUSG00000102735_Gm7369_Pr...,6.124617e-05,39.114555,3.792898e-04,-4.793652e+00,0.551022,-4.145355,0.757040,1.561609e-03,0.009976,0.897691,0.999989
49,1_4857826_4858825_ENSMUSG00000033813_Tcea1_Pro...,4.987745e-03,8.212569,9.303640e-04,-8.834212e-01,0.630707,-6.551575,0.039178,4.101265e-02,0.018834,0.919782,0.999989
65,1_5276826_5277825_ENSMUSG00000104352_Gm7182_Pr...,1.485950e-04,3.644471,8.552730e-04,-1.565795e-16,1.000000,-0.470432,0.727363,2.994123e-03,0.017785,1.000000,0.999989
116,1_7663826_7664825_ENSMUSG00000103903_Rps2.ps2_...,1.340251e-09,36.171022,2.019928e-08,-2.143125e+00,0.586292,-8.800108,0.186326,2.461682e-07,0.000005,0.906632,0.999989
145,1_9600826_9601825_ENSMUSG00000067879_Vxn_Prote...,1.306821e-03,85.172298,2.029107e-03,2.998090e+01,0.153992,-41.416610,0.235867,1.548113e-02,0.031900,0.868377,0.999989
...,...,...,...,...,...,...,...,...,...,...,...,...
117349,9_122399380_122400379_ENSMUSG00000093598_A7300...,1.433546e-04,19.067037,1.911405e-04,2.481516e+00,0.501691,-7.622284,0.220745,2.913849e-03,0.006180,0.885501,0.999989
117386,9_122845380_122846379_ENSMUSG00000107504_Gm355...,1.382645e-06,8.130908,6.053225e-07,-7.609391e-01,0.474199,-4.312506,0.019908,8.068676e-05,0.000080,0.880453,0.999989
117437,9_123215380_123216379_ENSMUSG00000035498_Cdcp1...,1.010689e-05,86.795414,9.391460e-05,3.302428e+01,0.043570,-37.356446,0.163884,3.859363e-04,0.003744,0.833933,0.999989
117446,9_123317380_123318379_ENSMUSG00000058492_Scp2....,1.580426e-06,9.208290,6.183233e-05,-2.552267e-01,0.875068,-0.361620,0.893893,8.967397e-05,0.002760,0.981777,0.999989


In [17]:
results_df_filtered_DEG_male[results_df_filtered_DEG_male["CpG"].str.contains("Myl")]

,CpG,pval model,coef_condition,pval_condition,coef_annotation,pval_annotation,coef_interaction,pval_interaction,fdr pval model,fdr pval_condition,fdr pval_annotation,fdr pval_interaction
50349,17_71004558_71005557_ENSMUSG00000024048_Myl12a...,0.000018,11.454319,0.000071,-1.238851,0.545013,-2.379083,0.48693,0.000595,0.003044,0.896171,0.999989
116562,9_110740380_110741379_ENSMUSG00000059741_Myl3_...,0.000245,67.835560,0.001492,-8.835469,0.579765,-2.900513,0.91312,0.004418,0.026004,0.905127,0.999989


In [18]:
#filter significant tests
results_df_filtered_DEG_pd = results_df_filtered[results_df_filtered["fdr pval_annotation"] <= 0.05]
results_df_filtered_DEG_pd

,CpG,pval model,coef_condition,pval_condition,coef_annotation,pval_annotation,coef_interaction,pval_interaction,fdr pval model,fdr pval_condition,fdr pval_annotation,fdr pval_interaction
20887,11_108342580_108343579_ENSMUSG00000000049_Apoh...,1.211550e-05,3.531007,0.356148,18.766466,9.879889e-07,-19.857652,0.000531,0.000443,0.754179,0.034444,0.834151
32542,14_21851505_21852504_ENSMUSG00000021773_Comtd1...,2.434666e-06,28.269169,0.011995,49.723560,2.557590e-06,-33.546490,0.026360,0.000127,0.104102,0.046118,0.999989
41637,15_102976433_102977432_ENSMUSG00000036139_Hoxc...,1.027847e-07,3.786320,0.824251,108.470982,8.331163e-09,-82.071816,0.001212,0.000010,0.959506,0.000503,0.999989
78972,4_145317725_145318724_ENSMUSG00000028602_Tnfrs...,1.367937e-05,1.608380,0.863675,43.575029,2.671119e-06,-45.859466,0.001032,0.000487,0.969242,0.046118,0.999989
84889,5_115564786_115565785_ENSMUSG00000041638_Gcn1_...,1.947771e-05,6.583505,0.548855,52.519826,1.619324e-06,-50.933782,0.001674,0.000640,0.870051,0.039141,0.999989
97863,7_43662239_43663238_ENSMUSG00000030474_Siglece...,1.330540e-05,14.573634,0.143895,47.966512,1.139998e-06,-57.220957,0.000150,0.000477,0.483652,0.034444,0.442810
98583,7_45833239_45834238_ENSMUSG00000053801_Grwd1_P...,6.837276e-08,81.931326,0.012204,202.234318,4.912848e-09,-181.459647,0.000146,0.000007,0.105444,0.000503,0.442810


In [19]:
#filter significant tests
results_df_filtered_DEG_inter = results_df_filtered[results_df_filtered["fdr pval_interaction"] <= 0.05]
results_df_filtered_DEG_inter

,CpG,pval model,coef_condition,pval_condition,coef_annotation,pval_annotation,coef_interaction,pval_interaction,fdr pval model,fdr pval_condition,fdr pval_annotation,fdr pval_interaction
37422,15_30629433_30630432_ENSMUSG00000115337_493057...,2.088181e-13,6.389188,1.443560e-14,3.107727e-01,0.414244,-6.020539,1.165311e-10,1.587241e-10,8.307823e-11,0.872012,0.000014
94681,6_149309210_149310209_ENSMUSG00000032712_Resf1...,6.400608e-10,14.897959,1.106787e-10,-2.057462e-01,0.871293,-14.874412,7.283366e-08,1.304483e-07,7.600171e-08,0.981096,0.004401
107447,8_75107116_75108115_ENSMUSG00000005410_Mcm5_Pr...,2.707212e-11,6.395009,3.138142e-12,-4.591202e-17,1.000000,-4.739820,1.042617e-06,8.866817e-09,5.267589e-09,1.000000,0.042003


In [20]:
# Filter models with significant F-test
#drop sex chromosomes
results_df = results_df[~results_df["CpG"].str.startswith("Y_")]
results_df = results_df[~results_df["CpG"].str.startswith("X_")]
sig_models = results_df[results_df["fdr pval model"] <= 0.05]

# Count how many have at least one significant coefficient
sig_condition = (sig_models["fdr pval_condition"] <= 0.05).sum()
sig_annotation = (sig_models["fdr pval_annotation"] <= 0.05).sum()
sig_interaction = (sig_models["fdr pval_interaction"] <= 0.05).sum()

# Count how many have none of the coefficients significant
none_sig = sig_models[
    (sig_models["fdr pval_condition"] > 0.05) &
    (sig_models["fdr pval_annotation"] > 0.05) &
    (sig_models["fdr pval_interaction"] > 0.05)
].shape[0]

print(f"Significant condition: {sig_condition}")
print(f"Significant annotation: {sig_annotation}")
print(f"Significant interaction: {sig_interaction}")
print(f"No individual coefficient significant: {none_sig}")


Significant condition: 8458
Significant annotation: 7
Significant interaction: 3
No individual coefficient significant: 6639


Model F-test is about the overall significance of the model.

Individual coefficient p-values are about specific predictors.

You can have a significant model even if no single coefficient passes significance — this explains the discrepancy in your counts.

In [21]:
#select significant bins from df_long
df_long.loc[results_df_filtered_DEG_inter.index]

,CpG,Sample,Expression,Condition,annotation
37422,15_30629433_30630432_ENSMUSG00000115337_493057...,M1_s10,0.0,female,central
94681,6_149309210_149310209_ENSMUSG00000032712_Resf1...,M1_s10,0.0,female,central
107447,8_75107116_75108115_ENSMUSG00000005410_Mcm5_Pr...,M1_s10,0.0,female,central


In [22]:
df1=df_long
df2=results_df_filtered
df_filtered = df1[df1["CpG"].isin(df2["CpG"])]
df_filtered

,CpG,Sample,Expression,Condition,annotation
8,1_3670826_3671825_ENSMUSG00000051951_Xkr4_Prot...,M1_s10,0.000000,female,central
27,1_4607826_4608825_ENSMUSG00000102735_Gm7369_Pr...,M1_s10,0.000000,female,central
28,1_4609826_4610825_ENSMUSG00000102735_Gm7369_Pr...,M1_s10,0.000000,female,central
38,1_4768826_4769825_ENSMUSG00000103922_Gm6123_Pr...,M1_s10,0.000000,female,central
46,1_4807826_4808825_ENSMUSG00000025903_Lypla1_Pr...,M1_s10,0.000000,female,central
...,...,...,...,...,...
3994026,9_123022380_123023379_ENSMUSG00000110945_Gm985...,M3_s36,10.585016,male,proximal-distal
3994046,9_123215380_123216379_ENSMUSG00000035498_Cdcp1...,M3_s36,88.529229,male,proximal-distal
3994055,9_123317380_123318379_ENSMUSG00000058492_Scp2....,M3_s36,17.320936,male,proximal-distal
3994060,9_123526380_123527379_ENSMUSG00000025240_Sacm1...,M3_s36,13.471839,male,proximal-distal


In [24]:
df2=results_df_filtered_DEG_male
df3=results_df_filtered_DEG_pd
df4=results_df_filtered_DEG_inter
# assuming your dataframe is called df
mean_expr = (
    df_filtered.groupby(["CpG", "Condition", "annotation"], as_index=False)["Expression"]
      .mean()
)
mean_expr = mean_expr[mean_expr["annotation"] != "exclude"]
# Assuming your dataframe is called df
mean_expr_wide = mean_expr.pivot_table(
    index="CpG", 
    columns=["Condition", "annotation"], 
    values="Expression"
)

# Flatten the MultiIndex columns (Condition + annotation)
mean_expr_wide.columns = [f"{cond}_{ann}" for cond, ann in mean_expr_wide.columns]
mean_expr_wide["assignment"] = np.select(
    [
        mean_expr_wide.index.isin(df2["CpG"]),
        mean_expr_wide.index.isin(df3["CpG"]),
        mean_expr_wide.index.isin(df4["CpG"])
    ],
    [
        "Condition",
        "Annotation",
        "Interaction"
    ],
    default="unassigned"
)
desired_order = ["Condition", "Annotation", "Interaction", "unassigned"]
mean_expr_wide["assignment"] = pd.Categorical(mean_expr_wide["assignment"], categories=desired_order, ordered=True)
mean_expr_wide = mean_expr_wide.sort_values("assignment")
mean_expr_wide

,female_central,female_proximal-distal,male_central,male_proximal-distal,assignment
CpG,,,,,
2_151798001_151799000_ENSMUSG00000083088_Gm14153_ProcessedPseudogene,1.277008,5.637447,12.198004,8.396942,Condition
3_34019173_34020172_ENSMUSG00000105535_Gm43077_lincRNA,2.765250,0.000000,19.025269,11.891912,Condition
3_34183173_34184172_ENSMUSG00000105739_Gm5708_ProcessedPseudogene,4.981168,5.564048,62.071038,57.702249,Condition
3_34351173_34352172_ENSMUSG00000092196_Gm38505_lincRNA,2.261848,2.199131,19.743331,10.912425,Condition
3_34679173_34680172_ENSMUSG00000105101_Gm43206_TEC,4.297534,6.967989,20.015635,16.977939,Condition
...,...,...,...,...,...
14_55820505_55821504_ENSMUSG00000023411_Nfatc4_ProteinCoding,41.202485,38.711339,88.011817,78.737628,unassigned
4_108625725_108626724_ENSMUSG00000102912_Gm20731_TEC,42.954579,36.375783,90.881185,75.120086,unassigned
4_108620725_108621724_ENSMUSG00000084968_Gm12743_ProcessedTranscript,224.375436,205.276539,70.251710,82.100961,unassigned


In [25]:
# Save to Excel
mean_expr_wide.to_excel("OLS_data_plot_TSS.xlsx", index=True)  # index=True to keep row indices

pd.read_excel("OLS_data_plot_TSS.xlsx", sheet_name="Sheet1", index_col=0)  # index_col=0 if you saved the index


,female_central,female_proximal-distal,male_central,male_proximal-distal,assignment
CpG,,,,,
2_151798001_151799000_ENSMUSG00000083088_Gm14153_ProcessedPseudogene,1.277008,5.637447,12.198004,8.396942,Condition
3_34019173_34020172_ENSMUSG00000105535_Gm43077_lincRNA,2.765250,0.000000,19.025269,11.891912,Condition
3_34183173_34184172_ENSMUSG00000105739_Gm5708_ProcessedPseudogene,4.981168,5.564048,62.071038,57.702249,Condition
3_34351173_34352172_ENSMUSG00000092196_Gm38505_lincRNA,2.261848,2.199131,19.743331,10.912425,Condition
3_34679173_34680172_ENSMUSG00000105101_Gm43206_TEC,4.297534,6.967989,20.015635,16.977939,Condition
...,...,...,...,...,...
14_55820505_55821504_ENSMUSG00000023411_Nfatc4_ProteinCoding,41.202485,38.711339,88.011817,78.737628,unassigned
4_108625725_108626724_ENSMUSG00000102912_Gm20731_TEC,42.954579,36.375783,90.881185,75.120086,unassigned
4_108620725_108621724_ENSMUSG00000084968_Gm12743_ProcessedTranscript,224.375436,205.276539,70.251710,82.100961,unassigned


In [ ]:
## Continue to heatmap_OLS_TSS.py for plot